# Retrieval Scores: Keeping Only Relevant Chunks

In rag-01, `similarity_search` always returned `k` chunks, even for a question the handbook can't answer.
It never said **how relevant** those chunks were.

**In this notebook:**

1. `similarity_search` → `similarity_search_with_score`
2. Compare the scores for an in-scope and an off-topic question.
3. Keep only the **relevant** chunks, using a score threshold.

**Series:** RAG-01 Basic RAG → **RAG-02 Scores** → RAG-03 Qdrant


## Step 0 — Setup


In [ ]:
# uv add --active langchain langchain-text-splitters langchain-community pypdf
# uv add --active langchain-ollama

In [ ]:
DATA_PATH = "shop_handbook.pdf"

CHAT_PROVIDER = "ollama"
CHAT_MODEL = "gemma3:1b"
EMBED_PROVIDER = "ollama"
EMBED_MODEL = "nomic-embed-text:latest"
TOP_K = 3

SYSTEM_PROMPT = (
    "You are a helpful Clause Shop assistant. "
    "Answer only from the provided context. "
    "If the answer is not in the context, say you don't know."
)

## Step 1 — Build the index (same as RAG-01)


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.embeddings import init_embeddings
from langchain_core.vectorstores import InMemoryVectorStore


pages = PyPDFLoader(DATA_PATH).load_and_split()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""],
)
chunks = text_splitter.split_documents(pages)

embedd = init_embeddings(model=EMBED_MODEL, provider=EMBED_PROVIDER)
vector_store = InMemoryVectorStore.from_documents(documents=chunks, embedding=embedd)

print(f"{len(pages)} pages -> {len(chunks)} chunks")

## Step 2 — `similarity_search`: documents only


In [ ]:
query = "How many days do I have to request a refund?"

docs = vector_store.similarity_search(query, k=TOP_K)   # [Document, ...]

for rank, doc in enumerate(docs, start=1):
    print(f"Rank {rank} | Page {doc.metadata['page'] + 1}")

We get 3 chunks, but is rank 1 a strong match or just the least bad one? **We can't tell.**

## Step 3 — `similarity_search_with_score`: documents and scores


In [ ]:
results = vector_store.similarity_search_with_score(query, k=TOP_K)   # [(Document, float), ...]

for rank, (doc, score) in enumerate(results, start=1):
    print(f"Rank {rank} | Score {score:.4f} | Page {doc.metadata['page'] + 1}")

For `InMemoryVectorStore`, the score is **cosine similarity**, so a **higher score means more relevant**. Qdrant with cosine distance (RAG-03) works the same way.

Some vector stores, such as FAISS by default, return a **distance** instead, where **lower** is better. Check which one your store uses before you set a threshold.


## Step 4 — In-scope vs off-topic


In [ ]:
def show(query, k=TOP_K):
    print(f"Q: {query}")
    for rank, (doc, score) in enumerate(vector_store.similarity_search_with_score(query, k=k), start=1):
        preview = " ".join(doc.page_content.split())[:70]
        print(f"  [{rank}] {score:.4f} | p{doc.metadata['page'] + 1} | {preview}")
    print()


show("How many days do I have to request a refund?")

In [ ]:
show("What is the capital of Pakistan?")

The off-topic question **still gets 3 chunks**, because vector search always returns its nearest neighbours. However, its scores are **lower**.

That difference is how we tell relevant chunks from irrelevant ones.


## Step 5 — Keep only relevant chunks

Choose a **threshold** between the two groups of scores you just saw. Chunks below it are dropped.
If none are left, answer "I don't know" **without calling the LLM**.

Adjust `SCORE_THRESHOLD` after looking at your own scores from Step 4.


In [ ]:
SCORE_THRESHOLD = 0.5   # adjust after looking at your scores in Step 4

In [ ]:
from langchain.chat_models import init_chat_model


llm = init_chat_model(model=CHAT_MODEL, model_provider=CHAT_PROVIDER, temperature=0.1)


def ask(query, k=TOP_K, threshold=SCORE_THRESHOLD):
    results = vector_store.similarity_search_with_score(query, k=k)
    relevant = [(doc, score) for doc, score in results if score >= threshold]

    if not relevant:
        print(f"AI: I don't know.  (best score {results[0][1]:.3f} < {threshold})")
        return

    docs = [doc for doc, _ in relevant]
    context = "\n\n".join(d.page_content for d in docs)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"},
    ]
    print("AI:", llm.invoke(messages).text)
    print("Used", len(docs), "relevant chunk(s), scores:", [round(s, 3) for _, s in relevant])


ask("How many days do I have to request a refund?")
print()
ask("What is the capital of Pakistan?")

## Summary

|                                   | `similarity_search` | `similarity_search_with_score` |
| --------------------------------- | ------------------- | ------------------------------ |
| Returns                           | `[Document, ...]`   | `[(Document, score), ...]`     |
| Shows how relevant each chunk is? | No                  | Yes                            |
| Can filter irrelevant chunks?     | No                  | Yes, with a threshold          |

- The threshold depends on the **embedding model**. If you change the model, check the scores again and adjust it.
- If it's **too high**, real questions get "I don't know". If it's **too low**, off-topic chunks reach the LLM.

**Next — RAG-03:** move the vectors to **Qdrant**. `similarity_search_with_score` works the same way there.
